# PALSAR-1 SAR Change Detection Tool
## 東日本大震災前後の地表変化可視化

- **データ**: Tellus API（JAXA PALSAR-1）
- **対象エリア**: 東北沿岸（宮城県周辺）
- **比較期間**: 震災前 2011-03-03 ／ 震災後 2011-04-08
- **使用技術**: Python / Tellus Traveler API / matplotlib / Pillow / NumPy


In [ ]:
import requests
import json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
%matplotlib inline

# Tellus API設定
TOKEN = "YOUR_TELLUS_API_TOKEN"  # ← ここにTellusトークンを入力
BASE_URL = "https://www.tellusxdp.com/api/traveler/v1"
HEADERS = {
    "Authorization": "Bearer " + TOKEN,
    "Content-Type": "application/json",
}


## 1. シーン検索

In [ ]:
def search_scene(datasets=None, intersects=None, query={}, sortby=None, paginate=None):
    url = "{}/data-search/".format(BASE_URL)
    payloads = {}
    if isinstance(datasets, str):
        url = "{}/datasets/{}/data-search/".format(BASE_URL, datasets)
    if intersects is not None:
        payloads["intersects"] = intersects
    if query is not None:
        payloads["query"] = query
    if isinstance(sortby, list):
        payloads["sortby"] = sortby
    if paginate is not None:
        payloads["paginate"] = paginate
    res = requests.post(url, headers=HEADERS, data=json.dumps(payloads))
    res.raise_for_status()
    return res.json()

# 東北沿岸エリア・PALSAR-1データセットID
PALSAR1_DATASET_ID = "8836ec9a-35b5-43c3-92be-263498061e91"
TOHOKU_BBOX = {
    "type": "Polygon",
    "coordinates": [[
        [140.8, 38.0], [141.6, 38.0],
        [141.6, 38.6], [140.8, 38.6],
        [140.8, 38.0]
    ]]
}

# 震災前後のシーンID
TOHOKU_OLD = "3e9d2089-6a2a-4780-bdbc-b0ff7cd71df6"  # 2011-03-03（震災前）
TOHOKU_NEW = "91f6da1d-c464-47e1-9b92-730537227123"  # 2011-04-08（震災後）

print("シーンID設定完了！")


## 2. 画像ダウンロード

In [ ]:
def get_file_info(dataset_id, data_id):
    url = "{}/datasets/{}/data/{}/files/".format(BASE_URL, dataset_id, data_id)
    res = requests.get(url, headers=HEADERS)
    res.raise_for_status()
    return res.json()

def get_download_url(dataset_id, data_id, file_id):
    url = "{}/datasets/{}/data/{}/files/{}/download-url/".format(
        BASE_URL, dataset_id, data_id, file_id)
    res = requests.post(url, headers=HEADERS)
    res.raise_for_status()
    return res.json()

# 震災前ダウンロード（file_id=5）
url_old = get_download_url(PALSAR1_DATASET_ID, TOHOKU_OLD, 5)["download_url"]
with open("tohoku_before.jpg", "wb") as f:
    f.write(requests.get(url_old).content)

# 震災後ダウンロード（file_id=2）
url_new = get_download_url(PALSAR1_DATASET_ID, TOHOKU_NEW, 2)["download_url"]
with open("tohoku_after.jpg", "wb") as f:
    f.write(requests.get(url_new).content)

print("ダウンロード完了！")


## 3. 比較・差分可視化

In [ ]:
im_old = Image.open("tohoku_before.jpg")
im_new = Image.open("tohoku_after.jpg")

old_gray = np.array(im_old.convert("L")).astype(float)
new_gray = np.array(im_new.convert("L")).astype(float)

# 有効範囲でトリミング
valid_cols = np.where(new_gray.mean(axis=0) > 5)[0]
valid_rows = np.where(new_gray.mean(axis=1) > 5)[0]
col_start, col_end = valid_cols[0], valid_cols[-1]
row_start, row_end = valid_rows[0], valid_rows[-1]

old_crop = old_gray[row_start:row_end, col_start:col_end]
new_crop = new_gray[row_start:row_end, col_start:col_end]
diff_crop = new_crop - old_crop

# 3枚並べて表示
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

axes[0].imshow(old_crop, cmap="gray")
axes[0].set_title("Before  2011-03-03", fontsize=13)
axes[0].axis("off")

axes[1].imshow(new_crop, cmap="gray")
axes[1].set_title("After  2011-04-08", fontsize=13)
axes[1].axis("off")

im_diff = axes[2].imshow(diff_crop, cmap="RdBu_r", vmin=-100, vmax=100)
axes[2].set_title("Change Detection\n(Red=Increase / Blue=Decrease)", fontsize=13)
axes[2].axis("off")
plt.colorbar(im_diff, ax=axes[2], fraction=0.046, pad=0.04)

plt.suptitle("PALSAR-1 SAR Change Detection\nTohoku Coast / Great East Japan Earthquake 2011-03-11", fontsize=15)
plt.tight_layout()
plt.savefig("tohoku_change_detection_final.png", dpi=150, bbox_inches="tight")
plt.show()
print("保存完了！")


## 4. 技術的考察・今後の改善点

### 結果の解釈
- **赤（Increase）**: 後方散乱強度が増加 → 瓦礫堆積・地表の粗面化の可能性
- **青（Decrease）**: 後方散乱強度が減少 → 建物消失・水没・泥による平滑化の可能性

### 技術的課題
- JPEGプレビューは圧縮されているためピクセル単位の精度に限界がある
- 2シーン間の撮影軌道・角度の違いによりノイズが発生
- 精度向上には生データ（IMG-HHファイル）と位置合わせ（コレジストレーション）処理が必要

### 今後の改善点
- 生SARデータを用いた干渉SAR（InSAR）解析の実装
- QPS研究所のSAR衛星データとの比較検証
